In [1]:
dataset_path = "/kaggle/input/dataset"

In [2]:
# !pip install transformers datasets torch rouge-score numpy pandas scikit-learn

In [3]:
# Phần 2: Import các thư viện
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import os

2025-07-24 10:23:29.050504: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753352609.229875      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753352609.281450      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:

# Phần 3: Định nghĩa lớp Dataset và hàm tải dữ liệu
class SummarizationDataset(Dataset):
    def __init__(self, texts, summaries, tokenizer, max_input_length=512, max_target_length=128):
        self.texts = texts
        self.summaries = summaries
        self.tokenizer = tokenizer
        self.max_input_length = max_input_length
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        summary = self.summaries[idx]

        input_encoding = self.tokenizer(
            text,
            max_length=self.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        target_encoding = self.tokenizer(
            summary,
            max_length=self.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        return {
            "input_ids": input_encoding["input_ids"].squeeze(),
            "attention_mask": input_encoding["attention_mask"].squeeze(),
            "labels": target_encoding["input_ids"].squeeze()
        }

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    print(f"Total rows in dataset: {len(df)}")

    invalid_rows = []
    for idx, row in df.iterrows():
        if not isinstance(row["full_text"], str) or not isinstance(row["summary"], str):
            invalid_rows.append((idx, row["full_text"], row["summary"]))
        elif row["full_text"].strip() == "" or row["summary"].strip() == "":
            invalid_rows.append((idx, row["full_text"], row["summary"]))

    if invalid_rows:
        print("Invalid rows found:")
        for idx, text, summary in invalid_rows:
            print(f"Row {idx}: full_text={text} (type: {type(text)}), summary={summary} (type: {type(summary)})")

    df = df.dropna(subset=["full_text", "summary"])
    df = df[df["full_text"].apply(lambda x: isinstance(x, str) and x.strip() != "")]
    df = df[df["summary"].apply(lambda x: isinstance(x, str) and x.strip() != "")]

    print(f"Rows after cleaning: {len(df)}")
    return df["full_text"].tolist(), df["summary"].tolist()

In [5]:
# Phần 4: Hàm huấn luyện mô hình ViT5
def train_vit5(file_path, output_dir="/kaggle/working/vit5_summarization", model_name="VietAI/vit5-base"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # Load tokenizer and model
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
    except Exception as e:
        print(f"Error loading model/tokenizer: {e}")
        print("Ensure you have the correct model name or access to private repo with `huggingface-cli login`")
        raise

    # Load dataset
    try:
        texts, summaries = load_dataset(file_path)
        print("Sample texts:", texts[:5])
        print("Sample summaries:", summaries[:5])
    except Exception as e:
        print(f"Error loading dataset: {e}")
        raise

    # Split dataset into train and validation
    train_texts, val_texts, train_summaries, val_summaries = train_test_split(
        texts, summaries, test_size=0.2, random_state=42
    )

    # Create datasets
    train_dataset = SummarizationDataset(train_texts, train_summaries, tokenizer)
    val_dataset = SummarizationDataset(val_texts, val_summaries, tokenizer)

    # Define training arguments
    try:
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=3,
            per_device_train_batch_size=2,
            per_device_eval_batch_size=2,
            warmup_steps=500,
            weight_decay=0.01,
            logging_dir="./logs",
            logging_steps=10,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            save_safetensors=False,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            load_best_model_at_end=True,
            fp16=False,  # Tắt FP16 để tránh lỗi gradient
            learning_rate=2e-5,  # Giảm learning rate
            gradient_accumulation_steps=2,  # Tích lũy gradient
            max_grad_norm=1.0,  # Gradient clipping thay cho gradient_clip_val
            report_to="none"  # Tắt W&B
        )
    except Exception as e:
        print(f"Error initializing TrainingArguments: {e}")
        raise

    # Initialize trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset
    )

    # Train the model
    try:
        trainer.train()
    except Exception as e:
        print(f"Error during training: {e}")
        raise

    # Evaluate ROUGE scores on validation set
    from rouge_score import rouge_scorer
    from statistics import mean

    print("Evaluating model on validation set...")
    rouge_scorer_instance = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    rouge_scores_list = []

    for i in range(len(val_texts)):
        # Generate summary for validation text
        inputs = tokenizer(
            val_texts[i],
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)

        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=128,
            num_beams=4,
            early_stopping=True
        )
        generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
        
        # Calculate ROUGE scores
        scores = rouge_scorer_instance.score(val_summaries[i], generated_summary)
        rouge_scores_list.append(scores)

    # Calculate average ROUGE scores
    rouge1_f1 = mean([score['rouge1'].fmeasure for score in rouge_scores_list])
    rouge2_f1 = mean([score['rouge2'].fmeasure for score in rouge_scores_list])
    rougeL_f1 = mean([score['rougeL'].fmeasure for score in rouge_scores_list])

    print("Average ROUGE Scores on Validation Set:")
    print(f"ROUGE-1 F1: {rouge1_f1:.4f}")
    print(f"ROUGE-2 F1: {rouge2_f1:.4f}")
    print(f"ROUGE-L F1: {rougeL_f1:.4f}")

    # Save the model and tokenizer
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Model and tokenizer saved to {output_dir}")

In [6]:


# Phần 5: Hàm tóm tắt văn bản
def summarize_text(text, model_dir = "/kaggle/working/vit5_summarization", max_length=128):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        model = AutoModelForSeq2SeqLM.from_pretrained(model_dir).to(device)
    except Exception as e:
        print(f"Error loading model/tokenizer: {e}")
        raise

    inputs = tokenizer(
        text,
        max_length=512,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_length,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary



In [7]:
!pip install transformers

In [8]:
!pip install rouge-score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=eaeee811e068c74923ed1c30e30f3867c08a29c6e610014b9408f0d70e3987a0
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [9]:
import transformers
# Phần 6: Chạy huấn luyện và tóm tắt
if __name__ == "__main__":
    dataset_path = "/kaggle/input/dataset/cleaned_dataset.csv"
    print("Transformers version:", transformers.__version__)
    train_vit5(dataset_path)

    sample_text = """
    Việt Nam là một quốc gia nằm ở khu vực Đông Nam Á, nổi tiếng với lịch sử lâu đời, văn hóa phong phú và cảnh quan thiên nhiên tuyệt đẹp. Từ những cánh đồng lúa bát ngát ở đồng bằng sông Cửu Long đến những dãy núi hùng vĩ ở miền Bắc, đất nước này thu hút hàng triệu du khách mỗi năm. Hà Nội, thủ đô của Việt Nam, là trung tâm văn hóa và chính trị, trong khi TP. Hồ Chí Minh là trung tâm kinh tế sôi động. Ẩm thực Việt Nam cũng là một điểm nhấn, với các món ăn như phở, bánh mì và gỏi cuốn được yêu thích trên toàn thế giới.
    """
    summary = summarize_text(sample_text)
    print("Summary:", summary)

Transformers version: 4.52.4
Using device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

Total rows in dataset: 15922
Rows after cleaning: 15922
Sample texts: ['Bức ảnh cho thấy một cặp đôi đang thưởng thức trà trong một nhà hàng. Thoạt nhìn, bức ảnh trông khá bình thường. Nhưng không phải vậy, bạn có thể phát hiện ra điểm sai trong hình không? Sẵn sàng chưa? Bạn chỉ có 10 giây để giải câu đố này! >> Xem đáp án Mộc Trà (Theo Jagran Josh)', 'Trong cuộc tiếp đón viên sứ Sài Thung của nhà Nguyên, ông đã ngồi yên cho kẻ thù chọc đầu đến chảy máu mà không hề thay đổi nét mặt. Khi về, Sài Thung ra tận cửa tiễn ông. Gợi ý:Sau khi qua đời dân gian đã suy tôn ông thành Đức Thánh Trần hay còn gọi là Cửu Thiên Vũ Đế. Tên thật của ông có 12 chữ cái, bắt đầu là chữ T và kết thúc là chữ N. >> Xem video hướng dẫn cách giải ô chữ tại đây >> Xem đáp án Hi', 'Câu đố trí tuệ hôm nay thách thức bạn tìm ra cô gái trong hình. Sẵn sàng chưa? Hãy giải ngay câu đố này! >> Xem đáp án Mộc Trà (Theo Jagran Josh)', "Chứng kiến cảnh tượng này cả nữ du khách người Nhật và nhóm bạn không khỏi vui mừng. T

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
1,0.613400,0.566273
2,0.563200,0.556091
3,0.505300,0.559254


Evaluating model on validation set...
Average ROUGE Scores on Validation Set:
ROUGE-1 F1: 0.5607
ROUGE-2 F1: 0.2457
ROUGE-L F1: 0.3715
Model and tokenizer saved to /kaggle/working/vit5_summarization
Summary: Hà Nội, thủ đô của Việt Nam, là trung tâm văn hóa, chính trị và kinh tế của Việt Nam, thu hút hàng triệu du khách mỗi năm.


In [10]:
# # Cập nhật output_dir khi lưu mô hình
# output_dir = "/kaggle/working/vit5_summarization"
# model.save_pretrained(output_dir)
# tokenizer.save_pretrained(output_dir)
# print(f"Model and tokenizer saved to {output_dir}")


In [11]:
sample_text = """
Kết luận nêu rõ:

Tại phiên họp ngày 17/7/2025, sau khi nghe và cho ý kiến về báo cáo của Ban Tổ chức Trung ương về tình hình, tiến độ thực hiện các nghị quyết, kết luận của Trung ương, Bộ Chính trị về sắp xếp tổ chức bộ máy và đơn vị hành chính từ ngày 11/7 đến ngày 16/7/2025 (Báo cáo số 424-BC/BTCTW, ngày 16/7/2025) và Báo cáo giám sát của Ủy ban Kiểm tra Trung ương, Bộ Chính trị, Ban Bí thư kết luận như sau:

1. Cơ bản thống nhất với Báo cáo về tình hình, tiến độ thực hiện các nghị quyết, kết luận của Trung ương, Bộ Chính trị về sắp xếp tổ chức bộ máy và đơn vị hành chính (từ ngày 11/7 đến ngày 16/7/2025) do Ban Tổ chức Trung ương trình. Bộ Chính trị, Ban Bí thư ghi nhận, đánh giá cao Đảng ủy Chính phủ, Đảng ủy Quốc hội, Đảng ủy Mặt trận Tổ quốc, các đoàn thể Trung ương, Ban Tổ chức Trung ương, Ủy ban Kiểm tra Trung ương, Văn phòng Trung ương Đảng, Bộ Công an, Bộ Nội vụ đã thực hiện tốt công tác lãnh đạo, chỉ đạo, hướng dẫn, theo dõi, nắm tình hình; biểu dương các cơ quan trong hệ thống chính trị, nhất là tại cấp tỉnh, cấp xã đã nêu cao tinh thần trách nhiệm, chủ động, tích cực tổ chức triển khai, khắc phục nhiều khó khăn, vướng mắc, bảo đảm bộ máy mới hoạt động thông suốt, ổn định, phục vụ nhân dân ngày càng tốt hơn.
"""
summary = summarize_text(sample_text)
print("Summary:", summary)

Summary: Bộ Chính trị, Ban Bí thư kết luận cơ bản thống nhất với Báo cáo của Ban Tổ chức Trung ương về tình hình, tiến độ thực hiện các nghị quyết, kết luận của Trung ương, Bộ Chính trị về sắp xếp tổ chức bộ máy và đơn vị hành chính từ ngày 11/7 đến ngày 16/7/2025.
